In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from sliced_wasserstein import sliced_wasserstein_distance
from c2st import c2st_knn, c2st_nn, c2st_rf

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/frequency_power_analysis/frequency_power_data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,29,34,42,43,52,60,62,67,69,80]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
def amplify_frequency_band(signal, sampling_rate, low_freq, high_freq, amplification_factor):
    # FFT
    freqs = np.fft.rfftfreq(len(signal), d=1/sampling_rate)
    fft_coeffs = np.fft.rfft(signal)

    # Amplify the specified frequency band
    band_mask = (freqs >= low_freq) & (freqs <= high_freq)
    fft_coeffs[band_mask] *= amplification_factor

    # Inverse FFT
    modified_signal = np.fft.irfft(fft_coeffs, n=len(signal))
    return modified_signal

In [ ]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/frequency_power_analysis/finetune_model_weights"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_pretrain_subject_index_{subject_index}_start_idx_{start_index}.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model

In [ ]:
cfg = load_config()

subject_index = cfg.dataset.test_subject_indices[1]
cfg.dataset.subject_index = subject_index
cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
cli_args = parse_args()
cfg = update_config(cfg, cli_args)
save_config(cfg)
all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
all_epochs = all_epochs[150:]
labels_raw = labels_raw[150:]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

# test with subject 2 how OOD samples are when we change power of a frequency band in all channels

## test if changing power works correctly

In [ ]:
sample_1_channel_1 = all_epochs[0,0]
psd, freqs = mne.time_frequency.psd_array_multitaper(sample_1_channel_1 , 1000, fmin=2, fmax=45, adaptive=True, low_bias=True, normalization='full', verbose=False)
plt.plot(freqs, 10 * np.log10(psd))

In [ ]:
perturbed_sample1 = amplify_frequency_band(sample_1_channel_1, 1000, 8, 12, 5)
psd, freqs = mne.time_frequency.psd_array_multitaper(perturbed_sample1 , 1000, fmin=2, fmax=45, adaptive=True, low_bias=True, normalization='full', verbose=False)
plt.plot(freqs, 10 * np.log10(psd))

## perturb all samples

In [ ]:
perturbed_samples = np.zeros_like(all_epochs)
for sample in range(all_epochs.shape[0]):
    for ch in range(all_epochs.shape[1]):
        perturbed_samples[sample,ch] = amplify_frequency_band(all_epochs[sample,ch], 1000, 8, 12, 5)
 
        

## check deviation of samples from the original distribution to other samples form the original distribution

In [ ]:
#first randomly split the original samples into two parts and test the SWD and c2st between them
np.random.seed(42)
indices = np.arange(all_epochs.shape[0])
np.random.shuffle(indices)
split = int(0.5 * len(indices))
indices_1 = indices[:split]
indices_2 = indices[split:]
# both sets need to have the same number of samples. THerefore, we take the minimum of the two
indices_1 = indices_1[:len(indices_2)]
indices_2 = indices_2[:len(indices_1)]


samples_1 = all_epochs[indices_1].reshape(len(indices_1), -1)
samples_2 = all_epochs[indices_2].reshape(len(indices_2), -1)
# convert to tensor
samples_1 = torch.tensor(samples_1, dtype=torch.float32).to(device)
samples_2 = torch.tensor(samples_2, dtype=torch.float32).to(device)



In [ ]:
print(sliced_wasserstein_distance(samples_1, samples_2, device=device, num_projections=50))
print(sliced_wasserstein_distance(samples_1, samples_2, device=device, num_projections=200))
print(sliced_wasserstein_distance(samples_1, samples_2, device=device, num_projections=500))
print(sliced_wasserstein_distance(samples_1, samples_2, device=device, num_projections=1000))

In [ ]:
print(c2st_knn(samples_1, samples_2, n_neighbors=5))
print(c2st_knn(samples_1, samples_2, n_neighbors=10))
print(c2st_knn(samples_1, samples_2, n_neighbors=20))
print(c2st_knn(samples_1, samples_2, n_neighbors=50))
print(c2st_knn(samples_1, samples_2, n_neighbors=100))

In [ ]:
print(c2st_nn(samples_1, samples_2))


In [ ]:
print(c2st_rf(samples_1, samples_2))


result for the classifiers seem to suggest that the classifieres cant distinguish between the samples. This is as desired as the samples come from the same distribution. Furthermore shuffling the samples potentially had the effect of eliminating any shifts in distribution during the time series.

## check deviation of samples from the original distribution to perturbed samples

In [ ]:
samples_original = all_epochs.reshape(all_epochs.shape[0], -1)
samples_perturbed = perturbed_samples.reshape(perturbed_samples.shape[0], -1)
# convert to tensor
samples_original = torch.tensor(samples_original, dtype=torch.float32).to(device)
samples_perturbed = torch.tensor(samples_perturbed, dtype=torch.float32).to(device)

In [ ]:
print(sliced_wasserstein_distance(samples_original, samples_perturbed, device=device, num_projections=50))
print(sliced_wasserstein_distance(samples_original, samples_perturbed, device=device, num_projections=200))
print(sliced_wasserstein_distance(samples_original, samples_perturbed, device=device, num_projections=500))
print(sliced_wasserstein_distance(samples_original, samples_perturbed, device=device, num_projections=1000))


In [ ]:
print(c2st_knn(samples_original, samples_perturbed, n_neighbors=5))
print(c2st_knn(samples_original, samples_perturbed, n_neighbors=10))
print(c2st_knn(samples_original, samples_perturbed, n_neighbors=20))
print(c2st_knn(samples_original, samples_perturbed, n_neighbors=50))
print(c2st_knn(samples_original, samples_perturbed, n_neighbors=100))


In [ ]:
print(c2st_nn(samples_original, samples_perturbed))

In [ ]:
print(c2st_rf(samples_original, samples_perturbed))

## check how deviation from original distribution scales with magnitude of perturbation for alpha frequency band

In [ ]:
def check_perturbation_effect(og_samples, pert_samples, device):
    samples_original = og_samples.reshape(og_samples.shape[0], -1)
    samples_perturbed = pert_samples.reshape(pert_samples.shape[0], -1)
    # convert to tensor
    samples_original = torch.tensor(samples_original, dtype=torch.float32).to(device)
    samples_perturbed = torch.tensor(samples_perturbed, dtype=torch.float32).to(device)

    results = {}

    results['swd_500'] = sliced_wasserstein_distance(samples_original, samples_perturbed, device=device, num_projections=500)
    #results['swd_1000'] = sliced_wasserstein_distance(samples_original, samples_perturbed, device=device, num_projections=1000)

    results['c2st_knn_5'] = c2st_knn(samples_original, samples_perturbed, n_neighbors=5)
    results['c2st_knn_10'] = c2st_knn(samples_original, samples_perturbed, n_neighbors=10)
    results['c2st_knn_20'] = c2st_knn(samples_original, samples_perturbed, n_neighbors=20)
    results['c2st_knn_50'] = c2st_knn(samples_original, samples_perturbed, n_neighbors=50)
    results['c2st_knn_100'] = c2st_knn(samples_original, samples_perturbed, n_neighbors=100)

    results['c2st_nn'] = c2st_nn(samples_original, samples_perturbed)

    results['c2st_rf'] = c2st_rf(samples_original, samples_perturbed)

    return results

 

## perturb all samples all channels for every frequency band and different amplification factors for the given frequency band

In [ ]:
freq_bands = {"delta" : (0.5, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
amplification_factors = [0.2,0.5,2,3,5,10]

In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:
#results_dict = {}

#for key, perturbed_samples in perturbed_samples_dict.items():
#    print(key)
#    results_dict[key] = check_perturbation_effect(all_epochs, perturbed_samples, device)

#print(results_dict)
#np.save("results_dict.npy", results_dict)

# test how the model prediction changes with the perturbed samples vs the original samples

## get prediction on original samples and perturbed samples

### original predictions

In [ ]:
cfg = load_config()
pred_label_original = np.zeros((all_epochs.shape[0]))
uncertainties_original = np.zeros((all_epochs.shape[0]))
input_shape_st = (60, 900)                
for i in tqdm(range(0, len(all_epochs))):
    start_index = i+100
    inputs = torch.from_numpy(all_epochs[i])
    inputs = inputs.to(device).float()
    inputs = inputs.unsqueeze(0)

    model = load_model(cfg, start_index=start_index, subject_index=subject_index)
    pred_mean, log_var = model(inputs)[:, 0], model(inputs)[:, 1]
    var = torch.exp(log_var)
    pred_label_original[i] = pred_mean.cpu().detach().numpy()
    uncertainties_original[i] = var.cpu().detach().numpy()




### predictions for all perturbed samples

In [ ]:
def get_predictions_on_perturbed_samples(perturbed_samples, device, subject_index):
    pred_label_perturbed = np.zeros((perturbed_samples.shape[0]))
    uncertainties_perturbed = np.zeros((perturbed_samples.shape[0]))  
    for i in tqdm(range(0, len(perturbed_samples))):
        start_index = i + 100
        inputs = torch.from_numpy(perturbed_samples[i])
        inputs = inputs.to(device).float()
        inputs = inputs.unsqueeze(0)

        model = load_model(cfg,start_index=start_index, subject_index=subject_index)
        pred_mean, log_var = model(inputs)[:, 0], model(inputs)[:, 1]
        var = torch.exp(log_var)
        pred_label_perturbed[i] = pred_mean.cpu().detach().numpy()
        uncertainties_perturbed[i] = var.cpu().detach().numpy()
    
    return pred_label_perturbed, uncertainties_perturbed

In [ ]:
#predictions_dict = {}
#uncertainties_dict = {}

#for key, perturbed_samples in perturbed_samples_dict.items():
#    pred_labels, uncertainties = get_predictions_on_perturbed_samples(perturbed_samples, device, subject_index)
#    predictions_dict[key] = pred_labels
#    uncertainties_dict[key] = uncertainties

#np.save("predictions_dict.npy", predictions_dict)
#np.save("uncertainties_dict.npy", uncertainties_dict)

# compare alignment of original predictions with perturbed predictions

In [ ]:
predictions_perturbed= np.load("predictions_dict.npy", allow_pickle=True).item()


In [ ]:
import pandas as pd


# Initialize a DataFrame to store the comparison results
comparison_df = pd.DataFrame(index=amplification_factors, columns=freq_bands.keys())

# Fill the DataFrame with the mean absolute difference between original and perturbed predictions
for band_name in freq_bands.keys():
    for factor in amplification_factors:
        key = f"{band_name}_{factor}"
        if key in predictions_perturbed:
            original_predictions = pred_label_original
            perturbed_predictions = predictions_perturbed[key]
            mean_abs_diff = np.median(original_predictions - perturbed_predictions)
            comparison_df.loc[factor, band_name] = mean_abs_diff

print(comparison_df)
# Plot the comparison results
fig, ax = plt.subplots(figsize=(12, 8))
comparison_df.plot(kind='bar', ax=ax)
ax.set_title('Median difference between Original and Perturbed Predictions')
ax.set_xlabel('Amplification Factor')
ax.set_ylabel('Median Difference')
plt.xticks(rotation=0)
plt.legend(title='Frequency Bands')
plt.savefig("median_absolute_difference_all_channels_per_frequency_band.png")

# Plot the comparison results
fig, axes = plt.subplots(len(amplification_factors), len(freq_bands), figsize=(20, 20))
for i, factor in enumerate(amplification_factors):
    for j, band_name in enumerate(freq_bands.keys()):
        key = f"{band_name}_{factor}"
        if key in predictions_perturbed:
            axes[i, j].scatter(pred_label_original, predictions_perturbed[key], alpha=0.5)
            axes[i, j].set_title(f'{band_name} {factor}')
            if i == len(amplification_factors) - 1:
                axes[i, j].set_xlabel('Original Predictions')
            if j == 0:
                axes[i, j].set_ylabel('Perturbed Predictions')

plt.tight_layout()
plt.show()

x = np.arange(len(pred_label_original))
# Plot the comparison results
fig, axes = plt.subplots(len(amplification_factors), len(freq_bands), figsize=(20, 20))
for i, factor in enumerate(amplification_factors):
    for j, band_name in enumerate(freq_bands.keys()):
        key = f"{band_name}_{factor}"
        if key in predictions_perturbed:
            #axes[i, j].scatter(pred_label_original, predictions_perturbed[key], alpha=0.5)
            axes[i, j].scatter(x, pred_label_original, alpha=0.5, c='blue', label=f'original, {np.median(pred_label_original):.2f}', s=4)
            axes[i, j].scatter(x, predictions_perturbed[key], alpha=0.5, c='red', label=f'perturbed, {np.median(predictions_perturbed[key]):.2f}', s=4)

            axes[i, j].set_title(f'{band_name} {factor}')
            if i == len(amplification_factors) - 1:
                axes[i, j].set_xlabel('trial')
            if j == 0:
                axes[i, j].set_ylabel('predictions')
            axes[i, j].legend()

the power in a frequency band over all channels seem to influence the final prediction of the model for the delta, alpha and the gamma band especially, with the predicted amplitude falling with power in the delta band and growing with power in the gamma, alpha band.

To make sure the predictions are not due to evaluation in extrapolation we checked how OOD the perturbed samples are with SWD and c2st. We also want to see how these are aligned to the uncertainty of the prediction output by the model

also check if the prediction for some samples decreases while the prediction for other samples increases. This may point to interaction-effects of the power in one frequency band with other features

At last,  no change in beta band until the 10x amplification may suggest what we only extrapolate in the 10x case for the beta band
 (or that change in prediction due to extrapolation and change in prediction due to the true different result may balance out?). Furthmore the severity of outliers increases a lot in higher amplification settings. This suggests that some trials are definitely more susceptible to changes in power, again pointing to more complex interaction effects.

# check how well measures of OODness and uncertainity output by the network are aligned

## first check how the OOD metrics introduced in the paper perform

In [ ]:

results_dict = np.load("results_dict.npy", allow_pickle=True).item()

In [ ]:
results_dict.keys()
for idx, band_name in enumerate(freq_bands.keys()):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(14, 4), sharex=True)
    
    # Plot SWD separately
    y_values_swd = [results_dict[f'{band_name}_{factor}']['swd_500'].item() for factor in amplification_factors]
    axs[0].plot(amplification_factors, y_values_swd, label='swd_500', color='blue')
    
    axs[0].set_title(f'SWD for {band_name} Band')
    axs[0].set_xlabel('Amplification Factor')
    axs[0].set_ylabel('SWD Value')


    
    # Plot other methods together

    for method in results_dict[f'{band_name}_0.2'].keys():
        if method != 'swd_500':
            y_values = [results_dict[f'{band_name}_{factor}'][method].item() for factor in amplification_factors]
            axs[1].plot(amplification_factors, y_values, label=method)
    
    axs[1].set_title(f'Evaluation Metrics for {band_name} Band')
    axs[1].set_xlabel('Amplification Factor')
    axs[1].set_ylabel('Metric Value')
    axs[1].legend()
    fig.savefig(f"OODness_{band_name}.png")
    fig.show()


the results seem to suggest that for amplification you can at maxium go a factor of ~3 and at minimum a factor of ~0.5(this needs to be rechecked)
Also as expected, the more we amplify we the power in a frequency band the greater the the SWD becomes. Furthermore the RF classifier seems to be most proficient and presumably closest to the bayes optimal classifier.

## compare the results to the predicted uncertainties for the perturbed samples

In [ ]:
uncertainties_perturbed = np.load("uncertainties_dict.npy", allow_pickle=True).item()

In [ ]:

# Initialize a DataFrame to store the comparison results
uncertainty_comparison_df = pd.DataFrame(index=amplification_factors, columns=freq_bands.keys())

# Fill the DataFrame with the mean absolute difference between original and perturbed uncertainties
for band_name in freq_bands.keys():
    for factor in amplification_factors:
        key = f"{band_name}_{factor}"
        if key in uncertainties_perturbed:
            original_uncertainties = uncertainties_original
            perturbed_uncertainties = uncertainties_perturbed[key]
            mean_abs_diff = np.median(original_uncertainties - perturbed_uncertainties)
            uncertainty_comparison_df.loc[factor, band_name] = mean_abs_diff

print(uncertainty_comparison_df)

# Plot the comparison results
fig, ax = plt.subplots(figsize=(12, 8))
uncertainty_comparison_df.plot(kind='bar', ax=ax)
ax.set_title('Median Absolute Difference between Original and Perturbed Uncertainties')
ax.set_xlabel('Amplification Factor')
ax.set_ylabel('Median Absolute Difference')

fig.savefig("median_absolute_difference_uncertainties_all_channels_per_frequency_band.png")


# Plot the comparison results
fig, axes = plt.subplots(len(amplification_factors), len(freq_bands), figsize=(20, 20))
for i, factor in enumerate(amplification_factors):
    for j, band_name in enumerate(freq_bands.keys()):
        key = f"{band_name}_{factor}"
        if key in uncertainties_perturbed:
            axes[i, j].scatter(uncertainties_original, uncertainties_perturbed[key], alpha=0.5)
            axes[i, j].set_title(f'{band_name} {factor}')
            if i == len(amplification_factors) - 1:
                axes[i, j].set_xlabel('Original Uncertainties')
            if j == 0:
                axes[i, j].set_ylabel('Perturbed Uncertainties')


x = np.arange(len(uncertainties_original))
# Plot the comparison results
fig, axes = plt.subplots(len(amplification_factors), len(freq_bands), figsize=(20, 20))
for i, factor in enumerate(amplification_factors):
    for j, band_name in enumerate(freq_bands.keys()):
        key = f"{band_name}_{factor}"
        if key in uncertainties_perturbed:
            axes[i, j].scatter(x, np.log(uncertainties_original), alpha=0.5, c='blue', label=f'original, {np.median(uncertainties_original):.2f}', s=4)
            axes[i, j].scatter(x, np.log(uncertainties_perturbed[key]), alpha=0.5, c='red', label=f'perturbed, {np.median(uncertainties_perturbed[key]):.2f}', s=4)

            axes[i, j].set_title(f'{band_name} {factor}')
            if i == len(amplification_factors) - 1:
                axes[i, j].set_xlabel('trial')
            if j == 0:
                axes[i, j].set_ylabel('log uncertainties')
            axes[i, j].legend()
            # axes[i, j].set_ylim(0, 1)   
fig.savefig("log_uncertaintiesper frequency band and amp factor across trials.png")

The first plot seems to suggest that increasing the power in some frequency bands like beta and alpha even decreases the uncertainity (since it is computed via original_uncertainity-perturbed_uncertainty)
, which is not in line with our measures of OODness that suggest that samples are more OOD for higher factors and that should reflect in in the predicted uncertainty.

In [ ]:
uncertainties_perturbed

# which frequency band perturbation leads to the biggest changes?